# Entrenamiento de modelos: dengue pediátrico

Este notebook prepara el dataset, documenta la limpieza y entrena los dos modelos solicitados: regresión logística y red neuronal artificial.

In [1]:
from pathlib import Path
import sys
import json

import joblib
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from data_processing import prepare_training_data

DATA_PATH = ROOT / "MASTER_CHART_F1000Research.xlsx"
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)
RANDOM_STATE = 42
pd.set_option("display.max_columns", 50)

## 1. Carga y revisión inicial

In [2]:
raw_df = pd.read_excel(DATA_PATH)
print(f"Dimensiones originales: {raw_df.shape}")
raw_df.head()

Dimensiones originales: (303, 25)


,GROUPS: DENGUE FEVER OR COMPLICATED DENGUE,AGE,GENDER,DURATION OF FEVER,HEADACHE,MYALGIA,ABDOMINAL PAIN,RASH,VOMITING,BREATHLESSNESS,BLEEDING,BP AT ADMISSION,ORGANOMEGALY,HB,PCV,LOWEST WBC,HIGHEST WBC,PLATELET COUNT,ALT,CREATININE,FERRITIN,IV FLUIDS,ANTIBIOTICS,ANTIPYRETICS,STEROIDS
0,2,2,1,3,2,1,2,2,1,2,2,2,2,14.2,46.0,3400,5100,30000,53.0,0.62,4167.0,1,1,2,1
1,2,2,1,4,2,1,2,2,2,2,2,1,2,12.8,38.6,3000,5300,43000,21.0,0.43,108.0,1,2,2,2
2,1,1,2,3,2,1,2,2,2,2,2,1,2,11.7,35.5,2300,2600,129000,12.0,0.34,212.0,1,1,1,2
3,1,2,2,5,2,1,2,2,2,2,2,1,2,14.1,42.1,1500,4800,70000,171.0,0.80,1429.0,1,1,1,1
4,1,2,2,3,2,1,2,2,2,2,2,1,2,13.3,41.0,8600,9100,184000,45.0,0.44,NaN,1,2,1,2


## 2. Limpieza y creación de variables

La función compartida aplica la misma transformación que utilizará Flask:

- Normalización de nombres de columnas.
- Eliminación de duplicados.
- Conversión de códigos binarios 1/2 a 1/0.
- Eliminación de columnas con al menos 30% de nulos o con una categoría dominante de al menos 95%.
- Creación de rango de WBC, relación HB/PCV, cantidad de síntomas y cantidad de tratamientos.
- La imputación de valores numéricos se realiza después dentro del pipeline, usando únicamente el conjunto de entrenamiento.

In [3]:
X, y, columns_to_drop, cleaning_summary = prepare_training_data(raw_df)
print(json.dumps(cleaning_summary, indent=2, default=str))
print("\nVariables finales:")
print(list(X.columns))
print("\nValores faltantes antes de imputar:")
print(X.isna().sum()[X.isna().sum() > 0])

{
  "initial_rows": 303,
  "duplicate_rows_removed": 1,
  "final_rows": 302,
  "columns_dropped": [
    "BREATHLESSNESS",
    "BLEEDING",
    "ORGANOMEGALY",
    "FERRITIN"
  ],
  "missing_values_after_cleaning": {
    "AGE": 0,
    "GENDER": 0,
    "DURATION_OF_FEVER": 0,
    "HEADACHE": 0,
    "MYALGIA": 0,
    "ABDOMINAL_PAIN": 0,
    "RASH": 0,
    "VOMITING": 0,
    "BP_AT_ADMISSION": 0,
    "HB": 0,
    "PCV": 0,
    "LOWEST_WBC": 0,
    "HIGHEST_WBC": 0,
    "PLATELET_COUNT": 0,
    "ALT": 1,
    "CREATININE": 1,
    "IV_FLUIDS": 0,
    "ANTIBIOTICS": 0,
    "ANTIPYRETICS": 0,
    "STEROIDS": 0,
    "WBC_RANGE": 0,
    "HB_PCV_RATIO": 0,
    "SYMPTOM_COUNT": 0,
    "TREATMENT_COUNT": 0
  },
  "class_counts": {
    "0": 239,
    "1": 63
  }
}

Variables finales:
['AGE', 'GENDER', 'DURATION_OF_FEVER', 'HEADACHE', 'MYALGIA', 'ABDOMINAL_PAIN', 'RASH', 'VOMITING', 'BP_AT_ADMISSION', 'HB', 'PCV', 'LOWEST_WBC', 'HIGHEST_WBC', 'PLATELET_COUNT', 'ALT', 'CREATININE', 'IV_FLUIDS', 'ANTIBIO

In [4]:
print("Distribución de la variable objetivo:")
print(y.value_counts().rename(index={0: "Dengue no complicado", 1: "Dengue complicado"}))
X.describe().T

Distribución de la variable objetivo:
GROUPS
Dengue no complicado    239
Dengue complicado        63
Name: count, dtype: int64


,count,mean,std,min,25%,50%,75%,max
AGE,302.0,0.198675,0.399665,0.000000,0.000000,0.00000,0.0000,1.000000
GENDER,302.0,0.529801,0.499939,0.000000,0.000000,1.00000,1.0000,1.000000
DURATION_OF_FEVER,302.0,3.976821,2.038531,1.000000,2.000000,4.00000,5.0000,10.000000
HEADACHE,302.0,0.304636,0.461017,0.000000,0.000000,0.00000,1.0000,1.000000
MYALGIA,302.0,0.301325,0.459595,0.000000,0.000000,0.00000,1.0000,1.000000
ABDOMINAL_PAIN,302.0,0.165563,0.372305,0.000000,0.000000,0.00000,0.0000,1.000000
RASH,302.0,0.052980,0.224366,0.000000,0.000000,0.00000,0.0000,1.000000
VOMITING,302.0,0.417219,0.493918,0.000000,0.000000,0.00000,1.0000,1.000000
BP_AT_ADMISSION,302.0,0.874172,0.332206,0.000000,1.000000,1.00000,1.0000,1.000000
HB,302.0,13.234768,1.437704,9.300000,12.400000,13.20000,14.0000,20.800000


## 3. División estratificada

Se reserva el 20% para prueba y se conserva la proporción de las dos clases.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print(f"Entrenamiento: {X_train.shape}; prueba: {X_test.shape}")

Entrenamiento: (241, 24); prueba: (61, 24)


## 4. Entrenamiento de los modelos

Ambos modelos reciben imputación por mediana y estandarización. La regresión logística usa pesos balanceados porque la clase de dengue complicado es minoritaria.

In [6]:
common_steps = [
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
]

logistic_model = Pipeline(common_steps + [
    ("classifier", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE))
])

neural_model = Pipeline(common_steps + [
    ("classifier", MLPClassifier(
        hidden_layer_sizes=(16, 8),
        activation="relu",
        alpha=0.001,
        learning_rate_init=0.001,
        max_iter=1500,
        early_stopping=True,
        validation_fraction=0.20,
        n_iter_no_change=30,
        random_state=RANDOM_STATE,
    ))
])

models = {"logistic": logistic_model, "neural": neural_model}
for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"{name}: entrenado")

logistic: entrenado
neural: entrenado


## 5. Evaluación en el conjunto de prueba

In [7]:
def evaluate_model(model, X_eval, y_eval):
    predictions = model.predict(X_eval)
    probabilities = model.predict_proba(X_eval)[:, 1]
    return {
        "accuracy": float(accuracy_score(y_eval, predictions)),
        "precision": float(precision_score(y_eval, predictions, zero_division=0)),
        "recall": float(recall_score(y_eval, predictions, zero_division=0)),
        "f1": float(f1_score(y_eval, predictions, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_eval, probabilities)),
        "confusion_matrix": confusion_matrix(y_eval, predictions, labels=[0, 1]).tolist(),
        "classification_report": classification_report(y_eval, predictions, target_names=["No complicado", "Complicado"], zero_division=0),
    }

metrics = {}
for name, model in models.items():
    metrics[name] = evaluate_model(model, X_test, y_test)
    print(f"\n=== {name} ===")
    print(metrics[name]["classification_report"])
    print("Matriz de confusión:")
    print(np.array(metrics[name]["confusion_matrix"]))
    print(f"ROC-AUC: {metrics[name]['roc_auc']:.3f}")


=== logistic ===
               precision    recall  f1-score   support

No complicado       0.95      0.83      0.89        48
   Complicado       0.58      0.85      0.69        13

     accuracy                           0.84        61
    macro avg       0.77      0.84      0.79        61
 weighted avg       0.87      0.84      0.85        61

Matriz de confusión:
[[40  8]
 [ 2 11]]
ROC-AUC: 0.960

=== neural ===
               precision    recall  f1-score   support

No complicado       0.81      0.92      0.86        48
   Complicado       0.43      0.23      0.30        13

     accuracy                           0.77        61
    macro avg       0.62      0.57      0.58        61
 weighted avg       0.73      0.77      0.74        61

Matriz de confusión:
[[44  4]
 [10  3]]
ROC-AUC: 0.471


## 6. Guardado de artefactos para Flask

Se guarda un único archivo con los pipelines, el orden de variables y la información de limpieza. Así la aplicación no vuelve a entrenar los modelos y evita inconsistencias entre notebook y página web.

In [8]:
bundle = {
    "models": models,
    "feature_columns": list(X.columns),
    "columns_to_drop": columns_to_drop,
    "summary": cleaning_summary,
    "test_metrics": metrics,
    "target_mapping": {0: "Dengue no complicado", 1: "Dengue complicado"},
    # La aplicación usa esta referencia para procesar GROUPS de archivos cargados.
    "encode_target": lambda series: series.map({1: 0, 2: 1}).fillna(series).astype(int),
}

# Las funciones lambda no siempre son serializables de forma portable. Se elimina
# del artefacto y Flask usa la misma función compartida del módulo.
bundle.pop("encode_target")
joblib.dump(bundle, MODEL_DIR / "model_bundle.joblib")
with open(MODEL_DIR / "metrics.json", "w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=2, ensure_ascii=False)
print(f"Artefactos guardados en: {MODEL_DIR}")

Artefactos guardados en: c:\1-Universidad\6to\Analitica de datos\Manuel\M3\models


In [9]:
# Verificación rápida con el primer paciente del conjunto de prueba
sample = X_test.iloc[[0]]
for name, model in models.items():
    prediction = int(model.predict(sample)[0])
    probability = float(model.predict_proba(sample)[0, 1])
    print(name, "=>", "Dengue complicado" if prediction else "Dengue no complicado", f"({probability:.2%})")

logistic => Dengue complicado (61.13%)
neural => Dengue no complicado (28.74%)
